# H4: Does each top-8 carrier's mean delay in 2025 differ from all other carriers combined?

Pre-registered: the top 8 carriers by total scheduled flights in 2025; for each, Welch's t of its completed flights' arrival delays against all other carriers' flights combined. Raw p-values, Bonferroni and Benjamini-Hochberg (FDR 5%) adjustments, and which conclusions flip.

In [ ]:
import pandas as pd
from IPython.display import Markdown, display

from delaystats import plotting, results
from delaystats import stats_helpers as sh
from delaystats.athena import run_query

ALPHA = 0.05  # pre-registered, two-sided (docs/ANALYSIS_PLAN.md)
SEED = 20261014  # fixed for every bootstrap
pd.set_option("display.float_format", "{:,.4f}".format)


def fmt_p(p):
    return "< 0.0001" if p < 1e-4 else f"{p:.4f}"


def table(rs):
    return pd.DataFrame([r.as_row() for r in rs]).set_index("test")

## Data

In [ ]:
volume = run_query("h4_carrier_volume_2025")
top8 = list(volume["reporting_airline"].head(8))
flights = run_query("h4_flights_2025")
print(f"2025 completed flights: {len(flights):,}; top 8 carriers by scheduled flights: {top8}")
volume.head(10)

## Eight tests

In [ ]:
delays = flights["arrdelay"].to_numpy(float)
carrier = flights["reporting_airline"].to_numpy()
h4 = [
    sh.welch_t(
        delays[carrier == c],
        delays[carrier != c],
        name=f"h4_welch_{c}",
        groups=(c, "all_other_carriers"),
    )
    for c in top8
]
corrections = sh.multiple_comparison_table(h4, alpha=ALPHA)
rows = results.frame("H4", h4).merge(corrections.drop(columns="p_raw"), on="test")
results.save_frame("H4", rows)
plotting.correction_table_plot(
    corrections,
    ALPHA,
    "h4_corrections",
    "H4: p-values per carrier, raw and corrected (higher bar = smaller p)",
)
rows.set_index("test")[
    [
        "estimate",
        "estimate_ci_low",
        "estimate_ci_high",
        "effect_size",
        "p",
        "p_bonferroni",
        "p_bh",
        "significant_raw",
        "significant_bonferroni",
        "significant_bh",
    ]
]

## Interpretation

In [ ]:
n_raw = int(corrections["significant_raw"].sum())
n_bonf = int(corrections["significant_bonferroni"].sum())
n_bh = int(corrections["significant_bh"].sum())
flipped = list(corrections.loc[corrections["flips_bonferroni"] | corrections["flips_bh"], "test"])
largest = max(h4, key=lambda r: abs(r.estimate))
small_g = sum(abs(r.effect_size) < 0.2 for r in h4)
display(
    Markdown(f"""
Eight Welch's t-tests at alpha = {ALPHA}, two-sided: **{n_raw}** carriers differ significantly from
the rest before correction, **{n_bonf}** after Bonferroni (family-wise error) and **{n_bh}** after
Benjamini-Hochberg (false discovery rate 5%).

- **Conclusions that flip under correction:** {", ".join(flipped) if flipped else "none: with this much data every difference is either clearly significant or clearly not, so correction changes no verdict"}.
- **Why this table matters:** testing eight carriers at alpha = 0.05 gives a 1 - 0.95^8 = 34% chance of at
  least one false positive if no carrier truly differed. The corrections are what make a list of
  "worse than average" carriers defensible.
- **Size, not just significance:** the largest difference is {largest.groups if hasattr(largest, "groups") else largest.name.removeprefix("h4_welch_")}
  at {largest.estimate:+.2f} minutes; {small_g} of 8 carriers have |Hedges' g| < 0.2 (a small effect
  even when p is tiny).
""")
)